In [ ]:
%pip install yfinance

In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import os

# 1. Setup
yesterday = (datetime.now() - timedelta(1)).strftime('%Y-%m-%d')
start_date = "2010-01-01"

# 2. DEFINE TICKERS
base_lkr = "USDLKR=X" # Our reliable anchor
# Minor currencies vs USD (These are 100% reliable tickers)
minor_vs_usd = {
    "EURUSD=X": "EUR",
    "GBPUSD=X": "GBP",
    "AUDUSD=X": "AUD",
    "CADUSD=X": "CAD",
    "JPYUSD=X": "JPY"
}
# Macro Drivers (Essential for a 'Meaningful' dataset)
macro_tickers = {
    "GC=F": "Gold_Price",
    "BZ=F": "Oil_Price",
    "DX-Y.NYB": "USD_Index"
}

print(f"Initializing Master Data Collection (LKR + Macro)...")

# 3. DOWNLOAD ALL RAW DATA
all_tickers = [base_lkr] + list(minor_vs_usd.keys()) + list(macro_tickers.keys())
raw_data = yf.download(all_tickers, start=start_date, end=yesterday)['Close']

# 4. CLEANING & MERGE PREP
raw_data.index.name = None # Remove index name to stop 'Date' conflict
raw_data = raw_data.ffill().bfill() # Fill weekend/holiday gaps

# Prepare macro features as a separate lookup table
macro_df = raw_data[list(macro_tickers.keys())].rename(columns=macro_tickers)
macro_df['Join_Key'] = macro_df.index

# 5. GENERATE DATA WITH TRIANGULATION MATH
data_list = []

# A. Add USD (Direct)
usd_df = pd.DataFrame({
    'Date': raw_data.index,
    'LKR_Rate': raw_data[base_lkr],
    'Currency': 'USD'
}).dropna()
# Join with Macro features
data_list.append(usd_df.merge(macro_df, left_on='Date', right_on='Join_Key', how='left'))

# B. Add Others (Math: MinorRate * USDLKR)
for t_id, name in minor_vs_usd.items():
    curr_df = pd.DataFrame({
        'Date': raw_data.index,
        'LKR_Rate': raw_data[t_id] * raw_data[base_lkr],
        'Currency': name
    }).dropna()
    # Join with Macro features
    data_list.append(curr_df.merge(macro_df, left_on='Date', right_on='Join_Key', how='left'))

# 6. CONSOLIDATE AND SAVE
master_df = pd.concat(data_list).drop(columns=['Join_Key'])

# Ensure path exists and save to raw folder
output_path = "../data/raw/LKR_Forex_Macro_Raw.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
master_df.to_csv(output_path, index=False)

print(f"\nMASTER COLLECTION SUCCESSFUL!")
print(f"Total Rows: {len(master_df)}")
print(f"Currencies: USD, EUR, GBP, AUD, CAD, JPY")
print(f"RAW Features: Date, Currency, LKR_Rate, Gold_Price, Oil_Price, USD_Index")
print(f"Saved to: {output_path}")

Initializing Master Data Collection (LKR + Macro)...


[*********************100%***********************]  9 of 9 completed


MASTER COLLECTION SUCCESSFUL!
Total Rows: 25218
Currencies: USD, EUR, GBP, AUD, CAD, JPY
RAW Features: Date, Currency, LKR_Rate, Gold_Price, Oil_Price, USD_Index
Saved to: ../data/raw/LKR_Forex_Macro_Raw.csv
